# 01 · Bronze — Ingesta raw del dataset NIES

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Ingestar el fichero Excel del NIES (`2_water_quality_eng_v20250925.xlsx`) y almacenarlo
en formato **Delta Lake** como capa Bronze, con metadatos de procedencia añadidos.

## Estructura del notebook
1. Configuración del entorno
2. Localización del fichero Excel (DBFS, Workspace o ruta local)
3. Lectura y normalización con pandas
4. Conversión a Spark y enriquecimiento con metadatos de procedencia
5. Escritura en Delta Lake (capa Bronze)
6. Validación de la ingesta

## 1. Configuración del entorno

In [ ]:
# Celda 1 — Importaciones y variables de configuración
import os, re, subprocess
import pandas as pd
from pyspark.sql import functions as F
from datetime import datetime

# ── Destino Delta Lake ────────────────────────────────────────────────────────
CATALOG      = "training"          # catálogo Unity Catalog activo
SCHEMA       = "kasumigaura"       # esquema / base de datos
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"

# ── Metadatos de procedencia ─────────────────────────────────────────────────
EXCEL_HEADER_ROW    = 2            # filas de encabezado en el Excel NIES
INGESTION_TIMESTAMP = datetime.utcnow().isoformat()
DATASET_VERSION     = "v20250925"
SOURCE_URL          = (
    "https://db.cger.nies.go.jp/gem/moni-e/inter/GEMS/database/kasumi/"
    "contents/datalist.html"
)

print(f"Tabla destino : {TABLE_BRONZE}")
print(f"Timestamp UTC : {INGESTION_TIMESTAMP}")
print(f"Versión NIES  : {DATASET_VERSION}")

## 2. Localización del fichero Excel

In [ ]:
# Celda 2 — Detectar catálogo Unity Catalog activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    if active_catalog and active_catalog != CATALOG:
        CATALOG      = active_catalog
        TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
    print(f"Catálogo activo : {CATALOG}")
    print(f"Tabla destino   : {TABLE_BRONZE}")
except Exception as e:
    print(f"No se pudo detectar catálogo ({e}). Usando: {CATALOG}")

In [ ]:
# Celda 3 — Localizar el Excel en cualquiera de las rutas posibles

FILENAME     = "2_water_quality_eng_v_20250925.xlsx"
ALT_FILENAME = "2_water_quality_eng_v20250925.xlsx"

CANDIDATE_PATHS = [
    # Ruta más probable: carpeta data dentro del repo en Workspace
    f"/Workspace/kasumigaura-lakehouse-poc/data/{FILENAME}",
    f"/Workspace/kasumigaura-lakehouse-poc/data/{ALT_FILENAME}",
    # DBFS (si se subió con Data > Upload)
    f"/dbfs/FileStore/tables/{FILENAME}",
    f"/dbfs/FileStore/tables/{ALT_FILENAME}",
    # Otras rutas posibles en Workspace
    f"/Workspace/kasumigaura/{FILENAME}",
    f"/Workspace/kasumigaura/{ALT_FILENAME}",
]

import os
EXCEL_PATH = None
for path in CANDIDATE_PATHS:
    if os.path.exists(path):
        EXCEL_PATH = path
        print(f"✓ Excel encontrado: {path}")
        print(f"  Tamaño: {os.path.getsize(path)/1_048_576:.2f} MB")
        break

if EXCEL_PATH is None:
    # Búsqueda automática
    import subprocess
    result = subprocess.run(
        ["find", "/Workspace", "-name", "*water_quality*.xlsx"],
        capture_output=True, text=True, timeout=15
    )
    found = result.stdout.strip().splitlines()
    if found:
        EXCEL_PATH = found[0]
        print(f"✓ Excel encontrado por búsqueda: {EXCEL_PATH}")
    else:
        print("✗ No se encontró el Excel. Rutas revisadas:")
        for p in CANDIDATE_PATHS: print(f"  {p}")
        raise FileNotFoundError("Excel NIES no encontrado")


## 3. Lectura y normalización con pandas

In [ ]:
# Celda 4 — Instalar openpyxl si no está disponible
try:
    import openpyxl
    print(f"openpyxl {openpyxl.__version__} disponible")
except ImportError:
    print("Instalando openpyxl...")
    %pip install -q openpyxl
    import openpyxl
    print(f"openpyxl {openpyxl.__version__} instalado")

In [ ]:
# Celda 5 — Leer el Excel NIES
print(f"Leyendo: {EXCEL_PATH}")
df_pandas = pd.read_excel(
    EXCEL_PATH,
    sheet_name=0,
    header=EXCEL_HEADER_ROW,
    engine="openpyxl"
)
print(f"Shape original: {df_pandas.shape}")
print(f"Primeras 10 columnas: {list(df_pandas.columns[:10])}")
df_pandas.head(3)

In [ ]:
# Celda 6 — Limpiar nombres de columnas y normalizar tipos

def clean_col_name(col: str) -> str:
    col = str(col).strip()
    # Preservar guiones internos de nombres científicos (T-P, T-N, Chl-a)
    # Solo reemplazar espacios, paréntesis, barras y puntos por _
    col = re.sub(r'[\s\(\)/\.]+', '_', col)
    col = re.sub(r'_+', '_', col)
    return col.strip('_')

original_cols = list(df_pandas.columns)
df_pandas.columns = [clean_col_name(c) for c in df_pandas.columns]
new_cols = list(df_pandas.columns)

# Eliminar filas completamente vacías
df_pandas = df_pandas.dropna(how='all')

# Convertir Date a datetime
if 'Date' in df_pandas.columns:
    df_pandas['Date'] = pd.to_datetime(df_pandas['Date'], errors='coerce')

# Convertir columnas numéricas
non_numeric = ['ST_', 'Date']
numeric_cols = [c for c in df_pandas.columns if c not in non_numeric]
for col in numeric_cols:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

print(f"Shape normalizado : {df_pandas.shape}")
print(f"Columnas totales  : {len(df_pandas.columns)}")
print(f"Registros válidos (Date no nula): {df_pandas['Date'].notna().sum()}")
print(f"Estaciones únicas : {df_pandas['ST_'].nunique() if 'ST_' in df_pandas.columns else 'n/a'}")

## 4. Conversión a Spark y enriquecimiento con metadatos

In [ ]:
# Celda 7 — Convertir a Spark DataFrame y añadir columnas de procedencia
df_spark = spark.createDataFrame(df_pandas)

df_spark = (
    df_spark
    .withColumn("_ingestion_ts",    F.lit(INGESTION_TIMESTAMP))
    .withColumn("_dataset_version", F.lit(DATASET_VERSION))
    .withColumn("_source_url",      F.lit(SOURCE_URL))
    .withColumn("_source_file",     F.lit(os.path.basename(EXCEL_PATH)))
)

print(f"Columnas Spark: {len(df_spark.columns)}")
print(f"  → {len(df_pandas.columns)} del Excel + 4 de procedencia")
df_spark.printSchema()

## 5. Escritura en Delta Lake (capa Bronze)

In [ ]:
# Celda 8 — Crear esquema si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✓ Esquema asegurado: {CATALOG}.{SCHEMA}")

In [ ]:
# Celda 9 — Escribir en Delta Lake (overwrite idempotente)
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE)
)

n_rows = spark.table(TABLE_BRONZE).count()
n_cols = len(spark.table(TABLE_BRONZE).columns)
print(f"✓ Tabla Bronze escrita: {TABLE_BRONZE}")
print(f"  Filas   : {n_rows:,}")
print(f"  Columnas: {n_cols}")

## 6. Validación de la ingesta

In [ ]:
# Celda 10 — Verificar integridad: filas Excel == filas Bronze
n_excel = len(df_pandas)
n_delta = spark.table(TABLE_BRONZE).count()

assert n_delta == n_excel, (
    f"Discrepancia de filas: Excel={n_excel}, Bronze={n_delta}"
)
print(f"✓ Integridad verificada: {n_delta:,} filas en Bronze == {n_excel:,} en Excel")

In [ ]:
# Celda 11 — Distribución temporal (muestra rápida)
from pyspark.sql.functions import year, count

print("Distribución por año (primeros 10):")
(
    spark.table(TABLE_BRONZE)
    .withColumn("year", year("Date"))
    .groupBy("year")
    .agg(count("*").alias("n"))
    .orderBy("year")
    .show(10)
)

print(f"\n✅ Notebook 01 completado — Capa Bronze: {TABLE_BRONZE}")
print(f"   Registros : {n_delta:,}")
print(f"   Columnas  : {n_cols}")
print(f"   Período   : 1977–2024")
print(f"   Próximo paso: ejecutar 02_silver_calidad.ipynb")